# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CSV_PATH = "../results/loo_summary_deg_200.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,spearman,pearson,precision,edistance_cells,edistance_latents,edistance_local,mixing_index
0,120,baseline,Endothelial,0.591499,0.665669,0.165,7.813582,NaN,10.562345,0.569879
1,120,baseline,Epithelial,0.472023,0.205826,0.030,6.224121,NaN,5.466480,0.925464
2,120,baseline,Fibroblast,0.404607,0.143257,0.150,8.479630,NaN,10.999038,0.626885
3,120,baseline,Myeloid,0.452335,0.396188,0.195,10.854124,NaN,14.149105,0.703882
4,120,baseline,T_cell,0.630919,0.612747,0.335,17.865647,NaN,20.198926,0.806997


In [2]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "edistance_local": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "MintFlow",
    "cellina (ablated)",
    "cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina (graph)',
       'CPA', 'scGen', 'MintFlow'], dtype=object)

In [3]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
per_slide = (
    df.groupby(["model_name", "sid"])[list(METRICS)]
    .mean()
    .reset_index()
)

agg = (
    per_slide.groupby("model_name")[list(METRICS)]
    .agg(["mean", "std"])
    .reindex(MODEL_ORDER)
)
agg

spearman             pearson           precision            \
                       mean       std      mean       std      mean       std   
model_name                                                                      
mean shift         0.525516  0.134023  0.499172  0.199214  0.205500  0.054236   
CPA                0.417255  0.140550  0.479703  0.165215  0.188167  0.037306   
scGen              0.288333  0.199959  0.267848  0.280129  0.220167  0.067668   
MintFlow           0.447833  0.233363  0.515796  0.266885  0.136800  0.022521   
cellina (ablated)  0.501433  0.166272  0.577975  0.194140  0.248833  0.047931   
cellina (graph)    0.378673  0.195418  0.340462  0.268659  0.251000  0.054603   
cellina            0.619810  0.169275  0.679346  0.183934  0.295833  0.049902   

                  edistance_local            
                             mean       std  
model_name                                   
mean shift              10.300824  2.885203  
CPA                      3.062305  1.442197  
scGen                    3.148171  1.692514  
MintFlow                      NaN       NaN  
cellina (ablated)        2.943469  1.324077  
cellina (graph)          7.616906  4.872663  
cellina                  3.186728  1.320515

In [4]:
def find_best(agg: pd.DataFrame) -> dict:
    """Return {metric: model_name_of_best} honouring direction."""
    best = {}
    for metric, direction in METRICS.items():
        means = agg[(metric, "mean")].dropna()
        if means.empty:
            continue
        best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s


best = find_best(agg)

table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])
for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        table.loc[model, col] = format_cell(mean, std, bold=(best.get(metric) == model))

table.index.name = "Method"
table

,Spearman $\uparrow$,Pearson $\uparrow$,Precision $\uparrow$,E-dist (local) $\downarrow$
Method,,,,
mean shift,0.526 $\pm$ 0.134,0.499 $\pm$ 0.199,0.206 $\pm$ 0.054,10.301 $\pm$ 2.885
CPA,0.417 $\pm$ 0.141,0.480 $\pm$ 0.165,0.188 $\pm$ 0.037,3.062 $\pm$ 1.442
scGen,0.288 $\pm$ 0.200,0.268 $\pm$ 0.280,0.220 $\pm$ 0.068,3.148 $\pm$ 1.693
MintFlow,0.448 $\pm$ 0.233,0.516 $\pm$ 0.267,0.137 $\pm$ 0.023,--
cellina (ablated),0.501 $\pm$ 0.166,0.578 $\pm$ 0.194,0.249 $\pm$ 0.048,\textbf{2.943 $\pm$ 1.324}
cellina (graph),0.379 $\pm$ 0.195,0.340 $\pm$ 0.269,0.251 $\pm$ 0.055,7.617 $\pm$ 4.873
cellina,\textbf{0.620 $\pm$ 0.169},\textbf{0.679 $\pm$ 0.184},\textbf{0.296 $\pm$ 0.050},3.187 $\pm$ 1.321


In [5]:
# Render as LaTeX (booktabs). escape=False keeps our \textbf and math arrows.
latex = table.to_latex(
    escape=False,
    column_format="l" + "c" * table.shape[1],
    caption=(
        "Leave-one-celltype-out performance (top 200 DEGs). For each slide we "
        "first average over the held-out cell types, then report mean $\\pm$ std "
        f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
    ),
    label="tab:loo_deg200",
)
latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
print(latex)

(OUT_DIR / "loo_summary_deg_200_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 200 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 6 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_deg200}
\begin{tabular}{lcccc}
\toprule
 & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & E-dist (local) $\downarrow$ \\
Method &  &  &  &  \\
\midrule
mean shift & 0.526 $\pm$ 0.134 & 0.499 $\pm$ 0.199 & 0.206 $\pm$ 0.054 & 10.301 $\pm$ 2.885 \\
CPA & 0.417 $\pm$ 0.141 & 0.480 $\pm$ 0.165 & 0.188 $\pm$ 0.037 & 3.062 $\pm$ 1.442 \\
scGen & 0.288 $\pm$ 0.200 & 0.268 $\pm$ 0.280 & 0.220 $\pm$ 0.068 & 3.148 $\pm$ 1.693 \\
MintFlow & 0.448 $\pm$ 0.233 & 0.516 $\pm$ 0.267 & 0.137 $\pm$ 0.023 & -- \\
cellina (ablated) & 0.501 $\pm$ 0.166 & 0.578 $\pm$ 0.194 & 0.249 $\pm$ 0.048 & \textbf{2.943 $\pm$ 1.324} \\
cellina (graph) & 0.379 $\pm$ 0.195 & 0.340 $\pm$ 0.269 & 0.251 $\pm$ 0.055 & 7.617 $\pm$ 4.873 \\
cellina & \textbf{0.620 $\pm

1128

In [6]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "edistance_local": "E-dist (local) ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method            | Spearman ↑        | Pearson ↑         | Precision ↑       | E-dist (local) ↓   |
|:------------------|:------------------|:------------------|:------------------|:-------------------|
| mean shift        | 0.526 ± 0.134     | 0.499 ± 0.199     | 0.206 ± 0.054     | 10.301 ± 2.885     |
| CPA               | 0.417 ± 0.141     | 0.480 ± 0.165     | 0.188 ± 0.037     | 3.062 ± 1.442      |
| scGen             | 0.288 ± 0.200     | 0.268 ± 0.280     | 0.220 ± 0.068     | 3.148 ± 1.693      |
| MintFlow          | 0.448 ± 0.233     | 0.516 ± 0.267     | 0.137 ± 0.023     | --                 |
| cellina (ablated) | 0.501 ± 0.166     | 0.578 ± 0.194     | 0.249 ± 0.048     | **2.943 ± 1.324**  |
| cellina (graph)   | 0.379 ± 0.195     | 0.340 ± 0.269     | 0.251 ± 0.055     | 7.617 ± 4.873      |
| cellina           | **0.620 ± 0.169** | **0.679 ± 0.184** | **0.296 ± 0.050** | 3.187 ± 1.321      |
